# Feature attribution
Full-cohort Integrated Gradients and held-out robustness analysis. Positive feature sums are the primary analysis; absolute attributions are used as a sensitivity analysis.

In [ ]:
from pathlib import Path
import sys
import torch
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_DIR))
from src import config
from src.attribution import aggregate_heldout, full_cohort_attribution, heldout_fold
from src.full_cohort import final_parameters

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
processed = torch.load(config.PROCESSED_DATA_PATH, map_location='cpu', weights_only=False)
print('Participants:', len(processed['participant_manifest']))

## Full-cohort attribution

In [ ]:
full_modality_summary, full_top_features = full_cohort_attribution(
    processed, final_parameters(config), config, DEVICE,
    config.OUTPUT_DIR / 'full_cohort_ig'
)
display(full_modality_summary)
display(full_top_features)

## Held-out attribution robustness

In [ ]:
checkpoint_dir = config.OUTPUT_DIR / 'nested_cv' / 'checkpoints'
heldout_dir = config.OUTPUT_DIR / 'heldout_ig'
for outer_fold in tqdm(range(1, config.OUTER_SPLITS + 1), desc='Held-out IG folds'):
    heldout_fold(
        processed, checkpoint_dir / f'outer_{outer_fold:02d}.pt', outer_fold,
        config, DEVICE, heldout_dir
    )
modality_summary, feature_summary, recurrent = aggregate_heldout(processed, heldout_dir)
display(modality_summary)
display(recurrent)